# Exercício 8 — cadeias **LCEL** compostas (LangChain)

O **LCEL** (*LangChain Expression Language*) compõe *runnables* com `|`, `RunnableParallel`, `RunnablePassthrough.assign`, `RunnableBranch` e `RunnableLambda`. Este caderno mostra padrões **juntos** — típicos em *pipelines* que não são ainda um grafo LangGraph.

**Pré-requisito:** exercício 5 (templates + `|`). **Variáveis:** `GOOGLE_API_KEY`, opcional `GEMINI_MODEL_EX08`, `GEMINI_MODEL_FALLBACKS`.

**Arranque Docker:** `./run.sh` nesta pasta.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

for p in (Path.cwd(), *Path.cwd().parents):
    env = p / ".env"
    if env.is_file():
        load_dotenv(env)
        break

_ex = Path.cwd().resolve().parent
if str(_ex) not in sys.path:
    sys.path.insert(0, str(_ex))
from lib_llm_fallback import gemini_model_candidates, make_gemini_chat_with_runtime_fallback

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no `.env` na raiz.")

_prim = (
    os.environ.get("GEMINI_MODEL_EX08")
    or os.environ.get("GEMINI_MODEL")
    or "gemini-2.0-flash"
).strip()
if _prim.removeprefix("models/").startswith("gemini-1.5"):
    print("Aviso: gemini-1.5* → gemini-2.0-flash")
    _prim = "gemini-2.0-flash"
candidatos = gemini_model_candidates(primary=_prim)
llm = make_gemini_chat_with_runtime_fallback(candidatos, temperature=0.25)
print("Modelos (ordem):", " → ".join(candidatos))

Modelos (ordem): gemini-2.0-flash → gemini-2.5-flash → gemini-2.0-flash-lite → gemini-2.5-flash-lite


## 1. `RunnableParallel` — vários ramos com o **mesmo** contexto

Corre **várias** sub-cadeias em paralelo sobre o mesmo dicionário de entrada (padrão *fan-out*). O resultado é um **dicionário** com uma chave por ramo — útil para recolher definição, prós e contras, etc., antes de fundir.


In [2]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

p_def = ChatPromptTemplate.from_template(
    "Define «{topico}» numa frase, em português europeu."
)
p_pros = ChatPromptTemplate.from_template(
    "Lista **duas** vantagens práticas de «{topico}» em bullets muito curtos."
)
p_cons = ChatPromptTemplate.from_template(
    "Lista **um** risco ou limitação de «{topico}» numa única frase."
)

parallel_tres_vias = RunnableParallel(
    definicao=(RunnablePassthrough() | p_def | llm | StrOutputParser()),
vantagens=(RunnablePassthrough() | p_pros | llm | StrOutputParser()),
limitacao=(RunnablePassthrough() | p_cons | llm | StrOutputParser()),
)

out = parallel_tres_vias.invoke({"topico": "RunnableParallel no LangChain"})
for k, v in out.items():
    print("---", k, "---")
    print(v[:400] if len(v) > 400 else v)

--- definicao ---
Em LangChain, `RunnableParallel` permite executar múltiplas cadeias de processamento (Runnables) em paralelo e combinar os seus resultados num único resultado final.
--- vantagens ---
Claro, aqui estão duas vantagens práticas do `RunnableParallel` no LangChain em bullets muito curtos:

*   **Execução paralela:** Executa múltiplas tasks simultaneamente, reduzindo o tempo total de processamento.
*   **Flexibilidade:** Permite combinar diferentes tipos de chains e prompts em paralelo, criando workflows complexos.
--- limitacao ---
RunnableParallel no LangChain pode sofrer de gargalos de desempenho se uma das tarefas paralelas for significativamente mais lenta que as outras, limitando o ganho de velocidade geral.


## 2. `RunnablePassthrough.assign` — enriquecer o **estado** passo a passo

Cada `assign` **adiciona chaves** ao dicionário sem apagar as anteriores (enquanto os ramos devolverem dict ou fundirem correctamente). Combina bem com paralelos para, no fim, ter `topico` + várias análises.


In [3]:
from langchain_core.runnables import RunnableLambda

# O resultado de `RunnableParallel` fica em `paralelo.*`; achatamos para o prompt de síntese.
def _flatten_para_sintese(d: dict) -> dict:
    p = d.get("paralelo") or {}
    return {
        "topico": d.get("topico", ""),
        "definicao": p.get("definicao", ""),
        "vantagens": p.get("vantagens", ""),
        "limitacao": p.get("limitacao", ""),
    }


cadeia_enriquecida = (
    RunnablePassthrough.assign(topico=lambda x: (x.get("topico") or "").strip())
    | RunnablePassthrough.assign(paralelo=parallel_tres_vias)
    | RunnableLambda(_flatten_para_sintese)
    | RunnablePassthrough.assign(
        sintese=(
            ChatPromptTemplate.from_template(
                "Tema: {topico}. Junta numa síntese curta (3 bullets) usando:\n"
                "- Definição:\n{definicao}\n\n- Vantagens:\n{vantagens}\n\n- Limitação:\n{limitacao}"
            )
            | llm
            | StrOutputParser()
        )
    )
)

final = cadeia_enriquecida.invoke({"topico": "orquestração de agentes com LangGraph"})
print("Chaves:", list(final.keys()))
print("\n--- síntese ---\n", final.get("sintese", ""))

Chaves: ['topico', 'definicao', 'vantagens', 'limitacao', 'sintese']

--- síntese ---
 Aqui está uma síntese curta sobre a orquestração de agentes com LangGraph:

*   **Definição:** LangGraph orquestra agentes de IA, permitindo que interajam e colaborem em fluxos de trabalho complexos para atingir um objetivo comum.
*   **Vantagens:** Aumenta a confiabilidade com tratamento de erros e loops, e oferece controle granular sobre a interação entre agentes para otimizar tarefas.
*   **Limitação:** A complexidade na configuração e depuração de fluxos de trabalho com múltiplos agentes pode dificultar a manutenção.


## 3. `operator.itemgetter` — mapear entradas para *prompts* (padrão oficial)

Em vez de `lambda`, usa-se `itemgetter` para extrair campos do dicionário — o mesmo estilo dos exemplos da documentação LangChain.


In [4]:
from operator import itemgetter

prompt_ctx = ChatPromptTemplate.from_messages(
    [
        ("system", "És conciso, em português europeu."),
        ("human", "Contexto: {contexto}\nPergunta: {pergunta}"),
    ]
)

chain_itemgetter = (
    {
        "contexto": itemgetter("contexto"),
        "pergunta": itemgetter("pergunta"),
    }
    | prompt_ctx
    | llm
    | StrOutputParser()
)

print(
    chain_itemgetter.invoke(
        {
            "contexto": "LCEL usa o operador | para compor runnables.",
            "pergunta": "Porque é que o resultado de A | B depende da ordem dos passos?",
        }
    )
)

Em LCEL, o operador `|` (pipe) sequencia a execução. O resultado de `A | B` depende da ordem porque o resultado de `A` é usado como entrada para `B`.  Se trocar a ordem para `B | A`, `B` receberá uma entrada diferente (ou nenhuma, se `B` for o primeiro passo), alterando o resultado final.


## 4. `RunnableBranch` — ramificação **condicional**

Útil para escolher sub-cadeias consoante o input (tamanho do texto, idioma, *intent*, etc.). O **último** argumento é o ramo **por omissão**. Cada condição é um *runnable* que avalia o input (tipicamente `RunnableLambda`).


In [5]:
from langchain_core.runnables import RunnableBranch

p_longo = ChatPromptTemplate.from_template(
    "O texto do utilizador é longo. Resume-o em 2 bullets: {texto}"
)
p_curto = ChatPromptTemplate.from_template(
    "O texto é curto. Reformula com uma pergunta de seguimento: {texto}"
)

branch_texto = RunnableBranch(
    (
        RunnableLambda(lambda d: len((d.get("texto") or "")) > 120),
        p_longo | llm | StrOutputParser(),
    ),
    p_curto | llm | StrOutputParser(),
)

print("--- curto ---")
print(branch_texto.invoke({"texto": "LangChain é uma biblioteca Python."}))
print("--- longo ---")
print(
    branch_texto.invoke(
        {
            "texto": "O LCEL permite compor chains com paralelismo e ramos. " * 5,
        }
    )[:500]
)

--- curto ---
LangChain é uma biblioteca Python? E quais são suas principais funcionalidades?
--- longo ---
*   O LangChain Expression Language (LCEL) facilita a criação de chains complexas.
*   Essas chains podem ser construídas com paralelismo e ramificações.


## 5. `RunnableLambda` — lógica Python pura dentro da *pipeline*

Encapsula funções (síncronas) como *runnable*: normalização, *feature flags*, validação leve, *chunking*, etc. Evita *lambdas* muito grandes — preferir funções nomeadas em módulos em produção.


In [6]:
def normalizar_pedido(d: dict) -> dict:
    q = (d.get("query") or "").strip()
    return {"query": q, "n_palavras": len(q.split())}


chain_norm = (
    RunnableLambda(normalizar_pedido)
    | RunnablePassthrough.assign(
        resposta=(
            ChatPromptTemplate.from_template(
                "Explica o termo «{query}» para quem tem {n_palavras} palavras no pedido (meta: 2 frases)."
            )
            | llm
            | StrOutputParser()
        )
    )
)

print(chain_norm.invoke({"query": "  memoização em grafos   "}))

{'query': 'memoização em grafos', 'n_palavras': 3, 'resposta': 'Memoização em grafos é a técnica de armazenar os resultados de cálculos para nós ou subproblemas já visitados. Isso evita recomputações desnecessárias, acelerando significativamente algoritmos que exploram o grafo.'}


## 6. Composição **final**: ramo + paralelo + fundo

1. Normaliza o tema.
2. Em paralelo: «explicação para leigo» vs «analógia técnica».
3. Junta num único parágrafo de conclusão.


In [7]:
p_leigo = ChatPromptTemplate.from_template(
    "Explica «{tema}» como a um colega **não** programador (max. 3 frases)."
)
p_tech = ChatPromptTemplate.from_template(
    "Dá uma **analogia técnica** (APIs, filas, ou grafos) para «{tema}» (max. 3 frases)."
)

duas_vozes = RunnableParallel(
    para_leigo=(RunnablePassthrough() | p_leigo | llm | StrOutputParser()),
    analogia_tech=(RunnablePassthrough() | p_tech | llm | StrOutputParser()),
)


def _merge_vozes(d: dict) -> dict:
    p = d.get("duas_vozes") or {}
    return {
        "tema": d.get("tema", ""),
        "para_leigo": p.get("para_leigo", ""),
        "analogia_tech": p.get("analogia_tech", ""),
    }


p_fecho = ChatPromptTemplate.from_template(
    "Integra os dois pontos de vista num **parágrafo** único:\n"
    "Vista leiga:\n{para_leigo}\n\nVista técnica:\n{analogia_tech}\n"
)

pipeline_final = (
    RunnablePassthrough.assign(tema=lambda x: (x.get("tema") or "").strip())
    | RunnablePassthrough.assign(duas_vozes=duas_vozes)
    | RunnableLambda(_merge_vozes)
    | RunnablePassthrough.assign(conclusao=(p_fecho | llm | StrOutputParser()))
)

res = pipeline_final.invoke({"tema": "LangGraph como máquina de estados para um agente"})
print(res.get("conclusao", ""))

LangGraph é uma ferramenta que permite construir agentes complexos, funcionando como um diagrama de fluxo (para o leigo) ou uma API de máquina de estados (para o técnico). Essencialmente, ele guia o agente (seja um robô ou um sistema de software) através de diferentes estados (como "pensar" ou "agir"), representados como nós, definindo as transições entre eles através de arestas condicionais. Uma fila de mensagens orquestra o fluxo de informações, garantindo que o agente avance de forma organizada e reativa, executando uma sequência de chamadas de API guiada por um grafo de transições, permitindo decisões complexas de forma estruturada e previsível.


## 7. Quando **parar** no LCEL e passar ao **LangGraph**

| Caso | Preferir |
|------|-----------|
| DAG simples, *fan-out*, condição 1 nível | LCEL (`RunnableParallel`, `RunnableBranch`, …) |
| Ciclos, *human-in-the-loop*, persistência de passos, *checkpoint* | LangGraph |
| Muitos estados e transição explícita | Grafo declarado |

O exercício **03** já usa LangGraph para **ReAct**; aqui o foco é **composição declarativa** sem vértices nomeados.
